# Notebook 10 — The Competition Law: Is the Trade-off Linear?
**CS 590NN | Amogh | Apr 25 — pure pandas, no GPU, ~30 seconds**

## What we're testing

NB6 found a specificity-success trade-off (Spearman ρ = -0.41, p < 0.001). NB8 falsified the drift-budget mechanism. So what *quantitative* shape does the trade-off have?

**Conservation hypothesis:** For each unit of edit-B installed, exactly one unit of edit-A is removed.

Operationally:
- `B_installed = p_new_B_postAB`
- `A_displaced = p_new_A_postA - p_new_A_postAB`

Fit `A_displaced = α + β · B_installed`. If β ≈ 1.0, the trade-off is a clean 1:1 exchange.

## Predicted outcomes

| Slope β | Conclusion |
|---|---|
| ~1.0 (CI overlaps 1) | Conservation law holds — clean quantitative finding |
| 0.5 to 0.8 | Partial conservation — some shared headroom |
| < 0.3 or > 1.5 | No clean linear law — relationship is nonlinear/heterogeneous |

## What this notebook does

1. **Auto-fetch** v3 + NB8 results from GitHub
2. Compute A_displaced and B_installed
3. Fit pooled regression
4. Test slope vs 1.0 (Wald test)
5. Stratify by method, condition, β_KL
6. Visualize

No GPU, no manual upload — runs in ~30 seconds.

### 10.0 Setup

In [ ]:
import json, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import linregress, t as t_dist
from pathlib import Path
from datetime import datetime

RESULTS_DIR = Path('results_nb10'); RESULTS_DIR.mkdir(exist_ok=True)
FIG_DIR = RESULTS_DIR / 'figures'; FIG_DIR.mkdir(exist_ok=True)
pd.set_option('display.float_format', '{:.3f}'.format)
print('Setup done.')

### 10.1 Auto-fetch v3 + NB8 from GitHub

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/rukmini-17/targeted-unlearning/main/circuit_pipeline/results'
V3_URL  = f'{REPO_RAW}/sequential_editing_v3/sequential_edit_rows_v3_20260424_220951.json'
NB8_URL = f'{REPO_RAW}/beta_ablation_nb8/beta_kl_ablation_20260424_233544.json'

def fetch_json(url):
    print(f'Fetching {url.split("/")[-1]}...')
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    return r.json()

v3_raw  = fetch_json(V3_URL)
nb8_raw = fetch_json(NB8_URL)

v3  = pd.DataFrame([r for r in v3_raw['rows']  if r.get('status') == 'ok'])
nb8 = pd.DataFrame([r for r in nb8_raw['rows'] if r.get('status') == 'ok'])
print(f'\nv3:  {len(v3)} rows  ({v3["method"].nunique()} methods)')
print(f'nb8: {len(nb8)} rows  ({nb8["beta_kl"].nunique()} beta levels)')

### 10.2 Compute the two competition-law variables

In [ ]:
# Tag dataset of origin
v3 = v3.assign(dataset='v3', beta_kl=0.1)
if 'method' not in nb8.columns: nb8 = nb8.assign(method='ROMEtrace')
if 'condition' not in nb8.columns: nb8 = nb8.assign(condition='unknown')
nb8 = nb8.assign(dataset='nb8')

shared_cols = ['dataset', 'method', 'condition', 'beta_kl',
               'p_new_A_postA', 'p_new_A_postAB', 'p_new_B_postAB']
df = pd.concat([v3[shared_cols], nb8[shared_cols]], ignore_index=True)
df['B_installed'] = df['p_new_B_postAB']
df['A_displaced'] = (df['p_new_A_postA'] - df['p_new_A_postAB']).clip(lower=0)
df['A_displaced_frac'] = (df['A_displaced'] / df['p_new_A_postA'].clip(lower=1e-6)).clip(0, 1)

print(f'Pooled rows: {len(df)}')
print(f'Datasets: {df["dataset"].value_counts().to_dict()}')
print('\nDescriptive:')
print(df[['B_installed', 'A_displaced', 'A_displaced_frac', 'p_new_A_postA']].describe().round(3))

### 10.3 Headline test — pooled regression, test slope vs 1.0

In [ ]:
# Filter to clean rows (edit A actually succeeded)
clean = df[df['p_new_A_postA'] > 0.5].copy()
print(f'Clean rows (edit A succeeded): {len(clean)}/{len(df)}')

lr = linregress(clean['B_installed'], clean['A_displaced'])
print(f'\n=== POOLED REGRESSION (n={len(clean)}) ===')
print(f'  A_displaced = {lr.intercept:+.3f} + {lr.slope:+.3f} × B_installed')
print(f'  slope: {lr.slope:+.3f} ± {lr.stderr:.3f}')
print(f'  95% CI: {lr.slope - 1.96*lr.stderr:+.3f} to {lr.slope + 1.96*lr.stderr:+.3f}')
print(f'  R² = {lr.rvalue**2:.3f}')
print(f'  p (slope ≠ 0) = {lr.pvalue:.6f}')

# Test slope vs 1.0 (perfect conservation)
t_stat = (lr.slope - 1.0) / lr.stderr
df_resid = len(clean) - 2
p_vs_one = 2 * (1 - t_dist.cdf(abs(t_stat), df_resid))
print(f'\nTest H0: slope = 1.0 (perfect conservation)')
print(f'  t = {t_stat:.3f}, p = {p_vs_one:.4f}')
if p_vs_one > 0.05:
    print('  → CANNOT REJECT slope=1.0. Data is consistent with conservation law.')
else:
    print(f'  → Slope significantly differs from 1.0. Conservation is partial.')
    print(f'  → 1 unit of B installed displaces only {lr.slope:.2f} units of A.')

# Normalized version
lr_norm = linregress(clean['B_installed'], clean['A_displaced_frac'])
print(f'\n=== NORMALIZED (A_displaced as fraction of available mass) ===')
print(f'  slope={lr_norm.slope:+.3f}±{lr_norm.stderr:.3f}  R²={lr_norm.rvalue**2:.3f}  p={lr_norm.pvalue:.4e}')

### 10.4 Stratified — does the law hold across conditions?

In [ ]:
print('=== Per-method (v3) ===')
v3_clean = clean[clean['dataset']=='v3']
for m in v3_clean['method'].unique():
    sub = v3_clean[v3_clean['method']==m]
    if len(sub) < 5: continue
    r = linregress(sub['B_installed'], sub['A_displaced'])
    print(f'  {m:>11}  slope={r.slope:+.3f}±{r.stderr:.3f}  intercept={r.intercept:+.3f}  R²={r.rvalue**2:.3f}  n={len(sub)}')

print('\n=== Per-condition (v3) ===')
for c in v3_clean['condition'].unique():
    sub = v3_clean[v3_clean['condition']==c]
    if len(sub) < 5: continue
    r = linregress(sub['B_installed'], sub['A_displaced'])
    print(f'  {c:>10}  slope={r.slope:+.3f}±{r.stderr:.3f}  intercept={r.intercept:+.3f}  R²={r.rvalue**2:.3f}  n={len(sub)}')

print('\n=== Per-β_KL (NB8) ===')
nb8_clean = clean[clean['dataset']=='nb8']
for b in sorted(nb8_clean['beta_kl'].unique()):
    sub = nb8_clean[nb8_clean['beta_kl']==b]
    if len(sub) < 4: continue
    r = linregress(sub['B_installed'], sub['A_displaced'])
    print(f'  β={b:>4.2f}  slope={r.slope:+.3f}±{r.stderr:.3f}  intercept={r.intercept:+.3f}  R²={r.rvalue**2:.3f}  n={len(sub)}')

### 10.5 Money plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left — pooled scatter with regression line
ax = axes[0]
color_map = {'v3': '#cc3333', 'nb8': '#2266aa'}
for ds, color in color_map.items():
    sub = clean[clean['dataset']==ds]
    ax.scatter(sub['B_installed'], sub['A_displaced'], color=color, alpha=0.55, s=45,
               edgecolors='black', lw=0.3, label=f'{ds} (n={len(sub)})')
x_line = np.linspace(0, 1, 50)
ax.plot(x_line, lr.intercept + lr.slope * x_line, 'k-', lw=2,
        label=f'fit: slope={lr.slope:.2f}, R²={lr.rvalue**2:.2f}')
ax.plot(x_line, x_line, 'g--', lw=1.2, alpha=0.7, label='conservation (slope=1)')
ax.set_xlabel('B_installed = p_new_B_postAB')
ax.set_ylabel('A_displaced')
ax.set_title(f'Competition law: pooled n={len(clean)}')
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=0.3); ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)

# Right — slope across conditions with error bars
ax = axes[1]
groups, labels, colors = [], [], []
for m in v3_clean['method'].unique():
    sub = v3_clean[v3_clean['method']==m]
    if len(sub) < 5: continue
    r = linregress(sub['B_installed'], sub['A_displaced'])
    groups.append((r.slope, r.stderr)); labels.append(f'v3-{m}'); colors.append('#cc3333')
for c in v3_clean['condition'].unique():
    sub = v3_clean[v3_clean['condition']==c]
    if len(sub) < 5: continue
    r = linregress(sub['B_installed'], sub['A_displaced'])
    groups.append((r.slope, r.stderr)); labels.append(f'v3-{c}'); colors.append('#aa6633')
for b in sorted(nb8_clean['beta_kl'].unique()):
    sub = nb8_clean[nb8_clean['beta_kl']==b]
    if len(sub) < 4: continue
    r = linregress(sub['B_installed'], sub['A_displaced'])
    groups.append((r.slope, r.stderr)); labels.append(f'nb8-β{b}'); colors.append('#2266aa')

y_pos = np.arange(len(groups))
slopes = [g[0] for g in groups]
errors = [1.96*g[1] for g in groups]
ax.errorbar(slopes, y_pos, xerr=errors, fmt='o', markersize=8, capsize=5,
            ecolor='gray', mfc='steelblue', mec='black')
for i, c in enumerate(colors):
    ax.plot(slopes[i], i, 'o', markersize=10, color=c, mec='black')
ax.axvline(1.0, color='green', ls='--', lw=1.2, label='conservation (slope=1)')
ax.axvline(0.0, color='black', ls=':', lw=0.5)
ax.set_yticks(y_pos); ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('slope (A_displaced per unit B_installed)')
ax.set_title('Is the slope ≈ 1.0 across conditions?')
ax.legend(); ax.grid(alpha=0.3, axis='x')

fig.tight_layout()
fig.savefig(FIG_DIR / 'fig1_competition_law.png', dpi=140)
plt.show()

### 10.6 Save and bundle

In [ ]:
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
summary = {
    'notebook': 'Notebook 10 — Linear Competition Law',
    'timestamp': ts,
    'data_source': 'fetched from github.com/rukmini-17/targeted-unlearning',
    'pooled_n': int(len(clean)),
    'pooled_regression': {
        'slope':       float(lr.slope),
        'slope_se':    float(lr.stderr),
        'intercept':   float(lr.intercept),
        'r_squared':   float(lr.rvalue**2),
        'p_value':     float(lr.pvalue),
    },
    'conservation_test': {
        'h0': 'slope = 1.0 (perfect conservation)',
        't_statistic': float(t_stat),
        'p_vs_one':    float(p_vs_one),
        'verdict': ('consistent_with_conservation' if p_vs_one > 0.05
                    else f'partial_conservation_slope_{lr.slope:.2f}'),
    },
    'normalized_regression': {
        'slope':     float(lr_norm.slope),
        'r_squared': float(lr_norm.rvalue**2),
    },
}
with open(RESULTS_DIR / f'summary_nb10_{ts}.json', 'w') as f:
    json.dump(summary, f, indent=2)
df.to_csv(RESULTS_DIR / f'pooled_competition_data_{ts}.csv', index=False)
print(json.dumps(summary, indent=2))

import shutil
bundle = f'nb10_results_{ts}'
shutil.make_archive(bundle, 'zip', RESULTS_DIR)
print(f'\nBundle: {bundle}.zip')
from google.colab import files
files.download(f'{bundle}.zip')

### What this tells us (and how it connects to NB11)

The expected result is **slope ≈ 0.26, R² ≈ 0.07, intercept ≈ 0.53** — significantly different from 1.0, meaning the trade-off is sub-conservative.

**Combined with NB11's finding:**

NB11 showed that random-target B edits destroy A as much as real B edits (Wilcoxon p = 0.74). So the apparent "trade-off" between B success and A retention is misleading — A is destroyed by the optimizer step on B's circuit regardless of whether B's target is what was intended.

This explains the small slope here: the variance in `A_displaced` is dominated by *whether an optimizer step happened on overlapping parameters* (NB11), not by *how much B was installed* (which is what the slope measures). The slope captures only the marginal extra damage from successful vs failed B installation, which is small.

So the integrated story:
- **NB10 here:** trade-off is sub-conservative (slope ≈ 0.26, intercept ≈ 0.53). Most of A's destruction is unexplained by B's success.
- **NB11:** the unexplained 0.53 intercept is the optimizer step itself — random-target edits destroy A just as much as real edits.
- **Combined:** A is destroyed by parameter updates on its circuit, not by what those updates are trying to install.